In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import roc_auc_score, f1_score, classification_report
import category_encoders as ce

In [4]:
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
df = train.merge(identity, how='left', on='TransactionID')
from sklearn.model_selection import train_test_split

X = df.drop(columns=["isFraud"])
y = df["isFraud"]


In [ ]:
!pip install mlflow dagshub --quiet

In [ ]:
import dagshub
dagshub.init(repo_owner='tsarc21', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)


# Feature Engineering

In [5]:
class InfCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        return X

In [6]:
class DropHighNaN(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = X.columns[X.isna().mean() > self.threshold]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [7]:
class DropIDColumns(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.cols_to_drop_ = [c for c in X.columns if "id" in c.lower()]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [8]:
class DropNearConstant(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.99):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = [
            col for col in X.columns
            if X[col].value_counts(normalize=True, dropna=False).iloc[0] > self.threshold
        ]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [9]:
class DropCorrelated(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        num_X = X.select_dtypes(include=np.number)
        corr = num_X.corr().abs()

        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

        self.cols_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [10]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class LogSkewTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=1.0):
        self.threshold = threshold

    def fit(self, X, y=None):
        num = X.select_dtypes(include=np.number)
        self.skew_cols_ = num.columns[num.skew().abs() > self.threshold]
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.skew_cols_:
            # replace inf just in case
            X[col] = X[col].replace([np.inf, -np.inf], np.nan)

            # fill NaN with 0 (safe baseline)
            X[col] = X[col].fillna(0)

            # handle negative values safely
            min_val = X[col].min()
            if min_val < 0:
                X[col] = X[col] - min_val  # shift to make all >= 0

            # now safe log transform
            X[col] = np.log1p(X[col])

        return X

In [11]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.rare_maps_ = {}

        cat_cols = X.select_dtypes(include="object").columns

        for col in cat_cols:
            freq = X[col].value_counts(normalize=True)
            self.rare_maps_[col] = freq[freq < self.threshold].index

        return self

    def transform(self, X):
        X = X.copy()

        for col, rare_vals in self.rare_maps_.items():
            X[col] = X[col].replace(rare_vals, "Other")

        return X

# Feature Selection

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.keep_columns_ = None

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        corr_matrix = X.corr().abs()

        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        to_drop = [
            column for column in upper.columns
            if any(upper[column] > self.threshold)
        ]

        self.keep_columns_ = [
            col for col in X.columns
            if col not in to_drop
        ]

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        return X[self.keep_columns_]

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import RFE


class RFESelector(BaseEstimator, TransformerMixin):
    def __init__(self, estimator, n_features_to_select=30):
        self.estimator = estimator
        self.n_features_to_select = n_features_to_select

    def fit(self, X, y):
        self.rfe_ = RFE(
            estimator=self.estimator,
            n_features_to_select=self.n_features_to_select
        )
        self.rfe_.fit(X, y)
        return self

    def transform(self, X):
        return self.rfe_.transform(X)

# Pipelines

In [13]:
import mlflow
mlflow.set_experiment("LogisticRegressionTraining")
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

mlflow.start_run(run_name="LogisticRegressionTraining")
X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

ModuleNotFoundError: No module named 'mlflow'

In [14]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [ ]:
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif

pre_cleaning = Pipeline(steps=[
    ("inf_clean", InfCleaner()),
    ("drop_nan", DropHighNaN()),
    ("drop_id", DropIDColumns()),
    ("drop_const", DropNearConstant()),
    ("log_skew", LogSkewTransformer()),
    ("rare_groups", RareCategoryGrouper()),
])

X_tmp = pre_cleaning.fit_transform(X_train)

num_cols = X_tmp.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_tmp.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("woe", ce.WOEEncoder())
])

feature_pipeline = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)

model = Pipeline(steps=[
    ("pre_clean", pre_cleaning),
    ("features", feature_pipeline),
    ("feature_selection", SelectKBest(score_func=f_classif, k=50)),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42
    ))
])

In [16]:
model.fit(X_train, y_train)

Pipeline(steps=[('pre_clean',
                 Pipeline(steps=[('inf_clean', InfCleaner()),
                                 ('drop_nan', DropHighNaN()),
                                 ('drop_id', DropIDColumns()),
                                 ('drop_const', DropNearConstant()),
                                 ('log_skew', LogSkewTransformer()),
                                 ('rare_groups', RareCategoryGrouper())])),
                ('features',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('woe',
                                                                   WOEEncoder())]),
                                                  ['ProductCD', 'card4',
                                                   'card6', 'P_emaildomain',
                                                   'R_emaildomain', 'M1', 'M2',
                                                   'M3', 'M4', 'M5', 'M6', 'M7',
                                                   'M8', 'M9', 'DeviceType',
                                                   'DeviceInfo'])])),
                ('feature_selection', SelectKBest(k=50)),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [17]:
y_pred = model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.82      0.89    455902
           1       0.10      0.57      0.17     16530

    accuracy                           0.81    472432
   macro avg       0.54      0.69      0.53    472432
weighted avg       0.95      0.81      0.87    472432



In [18]:
proba = model.predict_proba(X_valid)[:, 1]

thresholds = np.arange(0.05, 0.95, 0.01)

best_f1 = 0
best_t = 0

for t in thresholds:
    preds = (proba >= t).astype(int)
    f1 = f1_score(y_valid, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(best_t, best_f1)

0.8200000000000002 0.33880958793428495


# Leakage

In [ ]:
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif

pre_cleaning = Pipeline(steps=[
    ("inf_clean", InfCleaner()),
    ("drop_nan", DropHighNaN()),
    ("drop_id", DropIDColumns()),
    ("drop_const", DropNearConstant()),
    ("log_skew", LogSkewTransformer()),
    ("rare_groups", RareCategoryGrouper()),
])


X_clean = pre_cleaning.fit_transform(X_train)


num_cols = X_clean.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_clean.select_dtypes(include=["object", "category"]).columns.tolist()


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("woe", ce.WOEEncoder())
])


feature_pipeline = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)


model = Pipeline(steps=[
    ("pre_clean", pre_cleaning),
    ("features", feature_pipeline),
    ("corr_filter", CorrelationFilter(threshold=0.85)),
    ("feature_selection", SelectKBest(score_func=f_classif, k=50)),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight={0: 0.56, 1: 5.0},
        random_state=42
    ))
])

In [17]:
model.fit(X_train, y_train)

Pipeline(steps=[('pre_clean',
                 Pipeline(steps=[('inf_clean', InfCleaner()),
                                 ('drop_nan', DropHighNaN()),
                                 ('drop_id', DropIDColumns()),
                                 ('drop_const', DropNearConstant()),
                                 ('log_skew', LogSkewTransformer()),
                                 ('rare_groups', RareCategoryGrouper())])),
                ('features',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('woe',
                                                                   WOEEncoder())]),
                                                  ['ProductCD', 'card4',
                                                   'card6', 'P_emaildomain',
                                                   'R_emaildomain', 'M1', 'M2',
                                                   'M3', 'M4', 'M5', 'M6', 'M7',
                                                   'M8', 'M9', 'DeviceType',
                                                   'DeviceInfo'])])),
                ('corr_filter', CorrelationFilter()),
                ('feature_selection', SelectKBest(k=50)),
                ('clf',
                 LogisticRegression(class_weight={0: 0.56, 1: 5.0},
                                    max_iter=1000, random_state=42))])

# leakage

In [20]:
y_pred = model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    445659
           1       1.00      1.00      1.00     26773

    accuracy                           1.00    472432
   macro avg       1.00      1.00      1.00    472432
weighted avg       1.00      1.00      1.00    472432



In [29]:
y_pred = model.predict(X_valid)
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.96      0.97    113975
           1       0.30      0.47      0.37      4133

    accuracy                           0.94    118108
   macro avg       0.64      0.71      0.67    118108
weighted avg       0.96      0.94      0.95    118108



In [ ]:
mlflow.end_run()

In [ ]:
mlflow.set_experiment("LogisticRegressionTraining")

In [ ]:
y_pred = model.predict(X_valid)
y_proba = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, y_proba)
f1 = f1_score(y_valid, y_pred)
with mlflow.start_run(run_name="final_run"):
    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("f1_score", f1)

    report = classification_report(y_valid, y_pred, output_dict=True)
    for cls, metrics in report.items():
        if isinstance(metrics, dict):
            for m_name, val in metrics.items():
                mlflow.log_metric(f"{cls}_{m_name}", val)

    mlflow.sklearn.log_model(model, "LogisticRegressionTraining")
    
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/LogisticRegressionTraining"
    
    model_name = "fraud_detection_model"  
    model_registered = mlflow.register_model(model_uri, model_name)

    print(f"Model registered with name: {model_registered.name}, version: {model_registered.version}")